# Build sector return CSVs (`sector_rt_csv_2019_connected`)

Reproduces the per-sector daily-return matrices used to train the network DDPMs.

**Pipeline**
1. *(optional)* Collect 2019 daily close prices with `yfinance`.
2. Load close prices + company metadata + relationship adjacency.
3. For each sector, keep only **connected** companies (those with at least one
   relationship edge), compute daily returns `R_t = (P_{t+1} - P_t) / P_t × 100`,
   and write `sector_<Sector>_Rt.csv`.

Inputs are kept self-contained under `./inputs/` and
`./sector_relationship_adjacency/`. Outputs go to
`./sector_rt_csv_2019_connected/`.

> The `yfinance` step in section 1 is for reference / re-collection. It hits a
> live API, so prices may differ slightly from the frozen
> `inputs/DF_Stock_Close_2019_long.csv` that the rest of the notebook uses to
> exactly reproduce the released CSVs. Skip section 1 unless you want to refetch.

## 1. *(Optional)* Collect 2019 close prices with `yfinance`

Downloads adjusted close prices for every ticker in `DF_Stock_top5_linked.csv`
and writes them in the long format expected by section 2. Requires network
access and the `yfinance` package; skip if you already have
`inputs/DF_Stock_Close_2019_long.csv`.

In [1]:
# Set RUN_YFINANCE = True to (re)download prices. Default False -> use the frozen CSV.
RUN_YFINANCE = False

if RUN_YFINANCE:
    import datetime
    import time

    import pandas as pd
    import yfinance as yf
    from tqdm import tqdm

    META_PATH = "./inputs/DF_Stock_top5_linked.csv"
    OUT_LONG = "./inputs/DF_Stock_Close_2019_long.csv"
    START = datetime.datetime(2019, 1, 1)
    END = datetime.datetime(2019, 12, 31)

    meta = pd.read_csv(META_PATH)
    meta.columns = [c.strip() for c in meta.columns]
    tickers = (
        meta[["Company", "Stock_symbol"]].dropna()
        .drop_duplicates(subset=["Stock_symbol"]).reset_index(drop=True)
    )
    tickers["Stock_symbol"] = tickers["Stock_symbol"].astype(str).str.strip()

    def download(symbol, max_retry=3):
        for i in range(max_retry):
            try:
                df = yf.download(symbol, start=START, end=END, progress=False, auto_adjust=True)
                if not df.empty:
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = df.columns.get_level_values(0)
                    if "Close" in df.columns:
                        return df
            except Exception:
                pass
            time.sleep(0.5 * (i + 1))
        return None

    series, failed = [], []
    for _, row in tqdm(tickers.iterrows(), total=len(tickers)):
        df = download(row["Stock_symbol"])
        if df is None:
            failed.append(row["Stock_symbol"])
            continue
        series.append(df["Close"].rename(row["Stock_symbol"]))

    close_wide = pd.concat(series, axis=1).sort_index()
    close_long = (
        close_wide.reset_index()
        .melt(id_vars="Date", var_name="Stock_symbol", value_name="Close")
        .dropna(subset=["Close"])
        .merge(tickers, on="Stock_symbol", how="left")
    )
    close_long = close_long[["Date", "Company", "Stock_symbol", "Close"]].sort_values(["Date", "Stock_symbol"])
    close_long.to_csv(OUT_LONG, index=False)
    print(f"saved {OUT_LONG}: {len(close_long)} rows, {len(failed)} tickers failed")
else:
    print("RUN_YFINANCE = False -> using the frozen inputs/DF_Stock_Close_2019_long.csv")

RUN_YFINANCE = False -> using the frozen inputs/DF_Stock_Close_2019_long.csv


## 2. Load inputs

- `inputs/DF_Stock_Close_2019_long.csv` — long-format close prices (Date, Company, Stock_symbol, Close)
- `inputs/DF_Stock_top5_linked.csv` — metadata mapping Stock_symbol → Company, Sector_Broad
- `sector_relationship_adjacency/*.pickle` — per-sector relationship adjacency

Close prices are joined to sectors via the symbol metadata.

In [2]:
import os
import re
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

CLOSE_PATH = "./inputs/DF_Stock_Close_2019_long.csv"
META_PATH = "./inputs/DF_Stock_top5_linked.csv"
ADJ_DIR = Path("./sector_relationship_adjacency")
OUT_DIR = Path("./sector_rt_csv_2019_connected")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REL_NAMES = ["competitor", "customer", "investment", "partnership", "supplier"]


def load_pickle_compat(path):
    """Unpickle, tolerating numpy version differences (numpy._core)."""
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except ModuleNotFoundError as e:
        if "numpy._core" in str(e):
            import numpy.core
            sys.modules.setdefault("numpy._core", numpy.core)
            with open(path, "rb") as f:
                return pickle.load(f)
        raise


close_df = pd.read_csv(CLOSE_PATH)
meta_df = pd.read_csv(META_PATH)
close_df.columns = [c.strip() for c in close_df.columns]
meta_df.columns = [c.strip() for c in meta_df.columns]

close_df["Date"] = pd.to_datetime(close_df["Date"], errors="coerce")
close_df["Close"] = pd.to_numeric(close_df["Close"], errors="coerce")
close_df = close_df.dropna(subset=["Date", "Close", "Stock_symbol"])

meta_cols = meta_df[["Company", "Stock_symbol", "Sector_Broad"]].drop_duplicates()
df = close_df.merge(
    meta_cols[["Stock_symbol", "Sector_Broad", "Company"]],
    on="Stock_symbol", how="inner", suffixes=("_close", "_meta"),
)
df["Company"] = df["Company_meta"].fillna(df["Company_close"])

sectors = sorted(df["Sector_Broad"].dropna().unique().tolist())
print(f"sectors: {sectors}")

sectors: ['Consumer Discretionary', 'Financials', 'Health Care', 'Industrials', 'Information Technology']


## 3. Connected filtering + daily returns

For each sector: take the union of the relationship layers (undirected, no
self-loops), keep only companies with degree ≥ 1, pivot their close prices to a
Date × Company matrix, and compute `R_t = (P_{t+1} - P_t) / P_t × 100`. The last
day is dropped (no forward price). The result is saved as `sector_<Sector>_Rt.csv`.

In [3]:
def connected_companies(adj_payload):
    """Companies with at least one relationship edge (union over REL_NAMES)."""
    companies = adj_payload["companies"]
    rel_dict = adj_payload["adjacency_by_relationship"]
    n = len(companies)
    union = np.zeros((n, n), dtype=bool)
    for rel in REL_NAMES:
        if rel not in rel_dict:
            continue
        A = np.asarray(rel_dict[rel]) != 0
        A = np.logical_or(A, A.T)
        np.fill_diagonal(A, False)
        union |= A
    keep_idx = np.where(union.sum(axis=1) > 0)[0]
    return {companies[i] for i in keep_idx}


saved = []
for sector in sectors:
    safe_sector = re.sub(r"[^A-Za-z0-9_-]+", "_", sector.strip())

    adj_path = ADJ_DIR / f"sector_{sector}_adjacency_by_relationship.pickle"
    if not adj_path.exists():
        adj_path = ADJ_DIR / f"sector_{safe_sector}_adjacency_by_relationship.pickle"
    if not adj_path.exists():
        print(f"[skip] {sector}: adjacency file not found")
        continue

    keep = connected_companies(load_pickle_compat(adj_path))

    sec = df[(df["Sector_Broad"] == sector) & (df["Company"].isin(keep))]
    pivot = (
        sec.pivot_table(index="Date", columns="Company", values="Close", aggfunc="mean")
        .sort_index()
        .dropna(axis=1, how="all")
    )
    if pivot.empty:
        print(f"[skip] {sector}: empty pivot after connected filtering")
        continue

    rt = (pivot.shift(-1) - pivot) / pivot * 100.0
    rt = rt.replace([np.inf, -np.inf], np.nan).dropna(how="all")

    out_path = OUT_DIR / f"sector_{safe_sector}_Rt.csv"
    rt.to_csv(out_path, index_label="Date")
    saved.append(out_path)
    print(f"[save] {sector}: {out_path.name}  shape={rt.shape}  ({rt.shape[1]} companies)")

print(f"\nSaved {len(saved)} sector CSV(s) to {OUT_DIR}")

[save] Consumer Discretionary: sector_Consumer_Discretionary_Rt.csv  shape=(250, 22)  (22 companies)
[save] Financials: sector_Financials_Rt.csv  shape=(250, 31)  (31 companies)
[save] Health Care: sector_Health_Care_Rt.csv  shape=(250, 29)  (29 companies)
[save] Industrials: sector_Industrials_Rt.csv  shape=(250, 28)  (28 companies)
[save] Information Technology: sector_Information_Technology_Rt.csv  shape=(250, 42)  (42 companies)

Saved 5 sector CSV(s) to sector_rt_csv_2019_connected
